# Exp 2A – Multiclass Classification using DNN
**Dataset:** OCR Letter Recognition (OpenML – auto download)
**Aim:** Classify letters A-Z using Deep Neural Network

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical

print('Libraries imported!')

In [ ]:
# Load OCR Letter Recognition dataset (auto download)
print('Downloading dataset...')
letter = fetch_openml(name='letter', version=1, as_frame=True)

X = letter.data.astype(float)
y = letter.target

print(f'Shape: {X.shape}')
print(f'Features: {list(X.columns)}')
print(f'Classes: {sorted(y.unique())}')
print(f'\nClass distribution:')
print(y.value_counts().sort_index())

In [ ]:
# Class distribution bar chart
plt.figure(figsize=(16, 4))
y.value_counts().sort_index().plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Letter Class Distribution (A-Z)')
plt.xlabel('Letter'); plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout(); plt.show()

print('\nDataset Statistics:')
X.describe()

In [ ]:
# Encode labels A-Z to 0-25
le = LabelEncoder()
y_encoded = le.fit_transform(y)
num_classes = len(le.classes_)
print(f'Number of classes: {num_classes}')

# One-hot encode
y_cat = to_categorical(y_encoded, num_classes=num_classes)

# Train-test split
X_train, X_test, y_train, y_test, y_enc_train, y_enc_test = train_test_split(
    X, y_cat, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')

# Standardize
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test  = sc.transform(X_test)
print('Preprocessing done!')

In [ ]:
# Build DNN model
model = Sequential([
    Dense(256, activation='relu', input_dim=16),
    BatchNormalization(),
    Dropout(0.3),
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(num_classes, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
# Train
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
reduce_lr  = ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=5, verbose=1)

history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)
print('Training complete!')

In [ ]:
# Plot accuracy and loss
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'],     label='Train')
axes[0].plot(history.history['val_accuracy'], label='Val')
axes[0].set_title('Accuracy'); axes[0].legend(); axes[0].grid(True)

axes[1].plot(history.history['loss'],     label='Train')
axes[1].plot(history.history['val_loss'], label='Val')
axes[1].set_title('Loss'); axes[1].legend(); axes[1].grid(True)

plt.tight_layout(); plt.show()

In [ ]:
# Evaluate
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f'Test Accuracy : {acc*100:.2f}%')
print(f'Test Loss     : {loss:.4f}')

y_pred = np.argmax(model.predict(X_test), axis=1)
print('\nClassification Report:')
print(classification_report(y_enc_test, y_pred, target_names=le.classes_))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_enc_test, y_pred)
plt.figure(figsize=(16, 13))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Confusion Matrix – Letter Recognition')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.tight_layout(); plt.show()

In [ ]:
# Predict a random sample
idx = np.random.randint(0, len(X_test))
sample = X_test[idx].reshape(1, -1)
pred   = np.argmax(model.predict(sample))
conf   = model.predict(sample)[0].max() * 100

print(f'True Label      : {le.classes_[y_enc_test[idx]]}')
print(f'Predicted Label : {le.classes_[pred]}')
print(f'Confidence      : {conf:.2f}%')